# 01 - EDA: Khám phá dữ liệu ảnh NWPU-RESISC45

**Mục tiêu:** Phân tích thống kê mô tả bộ dữ liệu ảnh viễn thám trước khi tiền xử lý.

**Dataset:** NWPU-RESISC45 - 45 lớp cảnh viễn thám, ảnh 256×256 pixel.

## 0. Setup & Import

In [ ]:
import os, glob, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
from PIL import Image
import imagehash
from tqdm import tqdm
from collections import Counter, defaultdict
from scipy import stats
from pathlib import Path

warnings.filterwarnings('ignore')
plt.rcParams.update({
    'figure.figsize': (12, 6),
    'font.size': 11,
    'axes.titlesize': 13,
    'figure.dpi': 100
})
sns.set_style("whitegrid")

In [ ]:
TRAIN_DIR = r"D:\DataMining\Dataset\train"
TEST_DIR = r"D:\DataMining\Dataset\test"

classes = sorted(os.listdir(TRAIN_DIR))
print(f"Số lớp: {len(classes)}")
print(f"Danh sách: {', '.join(classes)}")

## 1. Tổng quan Dataset

In [ ]:
# Đếm ảnh mỗi lớp
train_counts = {}
test_counts = {}

for cls in classes:
    train_counts[cls] = len(glob.glob(os.path.join(TRAIN_DIR, cls, "*.jpg")))
    test_counts[cls] = len(glob.glob(os.path.join(TEST_DIR, cls, "*.jpg")))

df_counts = pd.DataFrame({
    'class': classes,
    'train': [train_counts[c] for c in classes],
    'test': [test_counts[c] for c in classes]
})
df_counts['total'] = df_counts['train'] + df_counts['test']

print(f"Tổng ảnh: {df_counts['total'].sum():,} (train: {df_counts['train'].sum():,}, test: {df_counts['test'].sum():,})")
print(f"Ảnh/lớp (train): min={df_counts['train'].min()}, max={df_counts['train'].max()}, mean={df_counts['train'].mean():.0f}")
print(f"Ảnh/lớp (test):  min={df_counts['test'].min()}, max={df_counts['test'].max()}, mean={df_counts['test'].mean():.0f}")

In [ ]:
# Bar chart phan bo anh theo lop
fig, ax = plt.subplots(figsize=(18, 5))
x = np.arange(len(classes))
ax.bar(x - 0.2, df_counts['train'], 0.4, label='Train', color='#4C72B0')
ax.bar(x + 0.2, df_counts['test'], 0.4, label='Test', color='#DD8452')
ax.set_xticks(x)
ax.set_xticklabels(classes, rotation=90, fontsize=7)
ax.set_ylabel('So anh')
ax.set_title('Phan bo so anh theo lop')
ax.legend()
plt.tight_layout()
plt.show()

# Nhan xet
n_unique_train = df_counts['train'].nunique()
n_unique_test = df_counts['test'].nunique()
print(f'Train: {n_unique_train} gia tri unique -> {"CAN BANG HOAN HAO" if n_unique_train == 1 else "Co chenh lech"}')
print(f'Test:  {n_unique_test} gia tri unique -> {"CAN BANG HOAN HAO" if n_unique_test == 1 else "Co chenh lech"}')
print(f"Ti le train:test = {df_counts['train'].iloc[0]}:{df_counts['test'].iloc[0]} = {df_counts['train'].iloc[0]//df_counts['test'].iloc[0]}:1")

In [ ]:
# Class imbalance ratio
imbalance_ratio = df_counts['train'].max() / df_counts['train'].min()
print(f'Class Imbalance Ratio (train): {imbalance_ratio:.2f}')
train_per_cls = df_counts['train'].iloc[0]
test_per_cls = df_counts['test'].iloc[0]
print(f'  -> Dataset CAN BANG HOAN HAO: moi lop deu co {train_per_cls} anh train, {test_per_cls} anh test')
print(f'  -> Khong can oversampling/undersampling')
print(f'  -> Cac khac biet giua lop (pixel, brightness, edge) la do NOI DUNG anh, khong phai do so luong')

### Ảnh mẫu toàn bộ 45 lớp

Hiển thị 1 ảnh đại diện từ mỗi lớp (theo thứ tự alphabet) để có cái nhìn tổng quan.

In [ ]:
# Grid ảnh mẫu - toàn bộ 45 lớp (5 hàng x 9 cột)
n_cols = 9
n_rows = (len(classes) + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, n_rows * 2.2))
for idx, cls in enumerate(classes):
    row, col = divmod(idx, n_cols)
    img_path = glob.glob(os.path.join(TRAIN_DIR, cls, "*.jpg"))[0]
    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    axes[row][col].imshow(img)
    axes[row][col].set_title(cls.replace('_', ' '), fontsize=7)
    axes[row][col].axis('off')
for idx in range(len(classes), n_rows * n_cols):
    row, col = divmod(idx, n_cols)
    axes[row][col].axis('off')
plt.suptitle("Ảnh mẫu từ toàn bộ 45 lớp (NWPU-RESISC45)", fontsize=14)
plt.tight_layout()
plt.show()

## 2. Pixel Distribution

In [ ]:
# Tính pixel stats cho toàn bộ 45 lớp (sample 50 ảnh/lớp)
SAMPLE_PER_CLASS = 50
all_means_r, all_means_g, all_means_b = [], [], []
class_pixel_stats = {}
per_class_r_means = {}

print("Đang tính pixel stats cho toàn bộ 45 lớp..")
for cls in tqdm(classes, desc="Pixel stats"):
    imgs = glob.glob(os.path.join(TRAIN_DIR, cls, "*.jpg"))[:SAMPLE_PER_CLASS]
    r_vals, g_vals, b_vals = [], [], []

    for p in imgs:
        img = cv2.imread(p)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        r_vals.append(img[:,:,0].mean())
        g_vals.append(img[:,:,1].mean())
        b_vals.append(img[:,:,2].mean())

    class_pixel_stats[cls] = {
        'r_mean': np.mean(r_vals), 'g_mean': np.mean(g_vals), 'b_mean': np.mean(b_vals),
        'r_std': np.std(r_vals),   'g_std': np.std(g_vals),   'b_std': np.std(b_vals)
    }
    per_class_r_means[cls] = r_vals
    all_means_r.extend(r_vals)
    all_means_g.extend(g_vals)
    all_means_b.extend(b_vals)

print(f"\nPixel Mean (toàn tập): R={np.mean(all_means_r):.1f}, G={np.mean(all_means_g):.1f}, B={np.mean(all_means_b):.1f}")
print(f"Pixel Std  (toàn tập): R={np.std(all_means_r):.1f}, G={np.std(all_means_g):.1f}, B={np.std(all_means_b):.1f}")

In [ ]:
# Histogram pixel mean per-image cho toàn bộ dataset
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
channels = [('R', all_means_r, '#e74c3c'), ('G', all_means_g, '#2ecc71'), ('B', all_means_b, '#3498db')]
for ax, (name, vals, color) in zip(axes, channels):
    ax.hist(vals, bins=50, color=color, alpha=0.7, edgecolor='white')
    ax.axvline(np.mean(vals), color='black', linestyle='--', label=f'Mean={np.mean(vals):.1f}')
    ax.set_title(f"Kênh {name}")
    ax.set_xlabel("Mean pixel value")
    ax.legend()
plt.suptitle("Phân bố pixel trung bình theo kênh (per-image, toàn bộ 45 lớp)", fontsize=13)
plt.tight_layout()
plt.show()

### Tổng quan pixel mean R/G/B theo lớp

Sắp xếp 45 lớp theo mean R pixel để thấy rõ sự khác biệt giữa các lớp.

In [ ]:
# Bar chart pixel mean R/G/B cho toàn bộ 45 lớp, sort theo R
df_pixel = pd.DataFrame({
    cls: {'R': s['r_mean'], 'G': s['g_mean'], 'B': s['b_mean']}
    for cls, s in class_pixel_stats.items()
}).T.sort_values('R')

fig, ax = plt.subplots(figsize=(18, 5))
x = np.arange(len(df_pixel))
w = 0.25
ax.bar(x - w, df_pixel['R'], w, label='R', color='#e74c3c', alpha=0.8)
ax.bar(x,     df_pixel['G'], w, label='G', color='#2ecc71', alpha=0.8)
ax.bar(x + w, df_pixel['B'], w, label='B', color='#3498db', alpha=0.8)
ax.set_xticks(x)
ax.set_xticklabels(df_pixel.index, rotation=90, fontsize=7)
ax.set_ylabel("Mean pixel value")
ax.set_title("Mean pixel R/G/B theo lớp (sắp xếp tăng dần theo R)")
ax.legend()
plt.tight_layout()
plt.show()

### Chọn 5 lớp đại diện cho so sánh pixel distribution chi tiết

Từ biểu đồ trên, ta chọn 5 lớp ở các vị trí: **min, Q1, median, Q3, max** theo R mean
để đại diện cho toàn bộ dải giá trị pixel.

In [ ]:
sorted_classes = df_pixel.index.tolist()
n = len(sorted_classes)
pick_idx = [0, n // 4, n // 2, 3 * n // 4, n - 1]
compare_classes = [sorted_classes[i] for i in pick_idx]

print("5 lớp đại diện (min -> Q1 -> median -> Q3 -> max theo R mean):")
for i, cls in enumerate(compare_classes):
    pos = ['min', 'Q1', 'median', 'Q3', 'max'][i]
    print(f"  [{pos:>6}] {cls}: R={class_pixel_stats[cls]['r_mean']:.1f}, G={class_pixel_stats[cls]['g_mean']:.1f}, B={class_pixel_stats[cls]['b_mean']:.1f}")

In [ ]:
# So sánh pixel distribution chi tiết giữa 5 lớp đại diện
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ch_idx, (ch_name, color) in enumerate(zip(['R', 'G', 'B'], ['#e74c3c', '#2ecc71', '#3498db'])):
    for cls in compare_classes:
        imgs = glob.glob(os.path.join(TRAIN_DIR, cls, "*.jpg"))[:30]
        vals = []
        for p in imgs:
            img = cv2.imread(p)
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            vals.append(img[:,:,ch_idx].ravel())
        all_px = np.concatenate(vals)
        axes[ch_idx].hist(all_px, bins=64, alpha=0.4, density=True, label=cls)
    axes[ch_idx].set_title(f"Kênh {ch_name}")
    axes[ch_idx].legend(fontsize=8)
    axes[ch_idx].set_xlabel("Pixel value")
plt.suptitle("So sánh phân bố pixel giữa 5 lớp đại diện (chọn từ sorted R mean)", fontsize=13)
plt.tight_layout()
plt.show()

### Kiểm định: Pixel trung bình giữa các lớp có khác biệt?

**H0:** Mean pixel R channel không khác biệt giữa 45 lớp.

**H1:** Ít nhất 1 cặp lớp có mean pixel R khác biệt.

**Bước:**
1. Kiểm tra normality (Shapiro-Wilk) trên vài lớp mẫu -> quyết định dùng parametric hay non-parametric
2. Levene test -> kiểm tra homogeneity of variance
3. ANOVA (parametric) + Kruskal-Wallis (non-parametric)
4. Post-hoc trên 5 lớp đại diện
5. Effect size (Eta-squared)

In [ ]:
# Bước 1: Normality check (Shapiro-Wilk trên 5 lớp đại diện)
print("1. Kiem tra normality (Shapiro-Wilk):")
print(f"   {'Class':<25} {'W-stat':>8} {'p-value':>10} {'Normal?':>10}")
print("   " + "-" * 55)
normality_ok = 0
for cls in compare_classes:
    w_stat, w_p = stats.shapiro(per_class_r_means[cls])
    is_normal = "Yes" if w_p > 0.05 else "No"
    if w_p > 0.05:
        normality_ok += 1
    print(f"   {cls:<25} {w_stat:>8.4f} {w_p:>10.4f} {is_normal:>10}")
print(f"\n   -> {normality_ok}/{len(compare_classes)} lop dat normality (alpha=0.05)")
if normality_ok >= 3:
    print("   -> Phan lon dat normality, ANOVA la phu hop. Kiem tra them Kruskal-Wallis de doi chieu.")
else:
    print("   -> Nhieu lop KHONG dat normality, uu tien Kruskal-Wallis (non-parametric).")

In [ ]:
# Bước 2: Levene test (homogeneity of variance)
groups = [per_class_r_means[cls] for cls in classes]
lev_stat, lev_p = stats.levene(*groups)
print(f"2. Levene test (homogeneity of variance):")
print(f"   Levene stat = {lev_stat:.2f}, p = {lev_p:.2e}")
if lev_p < 0.05:
    print("   -> Variance KHONG dong nhat. ANOVA van robust voi sample size bang nhau,")
    print("      nhung Kruskal-Wallis la kiem dinh an toan hon.")
else:
    print("   -> Variance dong nhat, ANOVA la phu hop.")

In [ ]:
# Bước 3: ANOVA + Kruskal-Wallis
f_stat, p_val = stats.f_oneway(*groups)
print(f"3a. One-Way ANOVA (R channel mean ~ 45 lop):")
print(f"    F = {f_stat:.2f}, p = {p_val:.2e}  {'-> BAC BO H0' if p_val < 0.05 else '-> KHONG BAC BO H0'}")

h_stat, p_val_kw = stats.kruskal(*groups)
print(f"\n3b. Kruskal-Wallis (non-parametric):")
print(f"    H = {h_stat:.2f}, p = {p_val_kw:.2e}  {'-> BAC BO H0' if p_val_kw < 0.05 else '-> KHONG BAC BO H0'}")

In [ ]:
# Bước 4: Post-hoc (Mann-Whitney pairwise trên 5 lớp đại diện, Bonferroni correction)
print("4. Post-hoc: Mann-Whitney U giua 5 lop dai dien (Bonferroni correction):")
n_comparisons = len(compare_classes) * (len(compare_classes) - 1) // 2
alpha_corrected = 0.05 / n_comparisons
print(f"   So cap so sanh: {n_comparisons}, alpha sau Bonferroni: {alpha_corrected:.4f}\n")

print(f"   {'Cặp':<40} {'U-stat':>10} {'p-value':>12} {'Ket qua':>15}")
print("   " + "-" * 79)
for i in range(len(compare_classes)):
    for j in range(i + 1, len(compare_classes)):
        c1, c2 = compare_classes[i], compare_classes[j]
        u_stat, u_p = stats.mannwhitneyu(per_class_r_means[c1], per_class_r_means[c2], alternative='two-sided')
        sig = "KHAC BIET" if u_p < alpha_corrected else "Khong khac biet"
        print(f"   {c1+' vs '+c2:<40} {u_stat:>10.1f} {u_p:>12.2e} {sig:>15}")

In [ ]:
# Bước 5: Effect size (Eta-squared)
all_r_flat = np.concatenate(groups)
grand_mean = all_r_flat.mean()
ss_between = sum(len(g) * (np.mean(g) - grand_mean)**2 for g in groups)
ss_total = np.sum((all_r_flat - grand_mean)**2)
eta_sq_pixel = ss_between / ss_total if ss_total > 0 else 0
print(f"5. Effect size:")
print(f"   Eta-squared = {eta_sq_pixel:.3f}")
if eta_sq_pixel > 0.14:
    print(f"   -> Effect size LON (>{0.14}): lop anh giai thich {eta_sq_pixel*100:.1f}% variance cua pixel mean")
elif eta_sq_pixel > 0.06:
    print(f"   -> Effect size TRUNG BINH")
else:
    print(f"   -> Effect size NHO")

print(f"\n=> KET LUAN: Pixel mean R co khac biet {'CO Y NGHIA' if p_val < 0.05 else 'KHONG Y NGHIA'} giua cac lop")
print(f"   (ANOVA F={f_stat:.2f}, Kruskal H={h_stat:.2f}, Eta²={eta_sq_pixel:.3f})")

## 3. Duplicate Detection (pHash)

In [ ]:
print("Đang tính pHash cho toàn bộ ảnh train..")
hash_dict = {}
hash_list = []

for cls in tqdm(classes, desc="pHash"):
    imgs = glob.glob(os.path.join(TRAIN_DIR, cls, "*.jpg"))
    for p in imgs:
        h = imagehash.phash(Image.open(p))
        fname = os.path.basename(p)
        hash_key = str(h)
        if hash_key not in hash_dict:
            hash_dict[hash_key] = []
        hash_dict[hash_key].append((cls, fname))
        hash_list.append((h, cls, fname, p))

exact_dupes = {k: v for k, v in hash_dict.items() if len(v) > 1}
print(f"\nExact pHash duplicates: {len(exact_dupes)} nhóm")

if exact_dupes:
    print("Ví dụ:")
    for k, v in list(exact_dupes.items())[:3]:
        print(f"  Hash {k}: {v}")

In [ ]:
# Near-duplicate detection (Hamming distance <= 4)
print("Near-duplicate scan (sample 2000 ảnh, threshold=4)..")
np.random.seed(42)
sample_idx = np.random.choice(len(hash_list), min(2000, len(hash_list)), replace=False)
sample_hashes = [hash_list[i] for i in sample_idx]

near_dupes = []
for i in range(len(sample_hashes)):
    for j in range(i+1, len(sample_hashes)):
        dist = sample_hashes[i][0] - sample_hashes[j][0]
        if dist <= 4:
            near_dupes.append((sample_hashes[i], sample_hashes[j], dist))

print(f"Near-duplicates tìm được: {len(near_dupes)} cặp (trong sample)")

if near_dupes:
    fig, axes = plt.subplots(min(3, len(near_dupes)), 2, figsize=(6, 3*min(3, len(near_dupes))))
    if len(near_dupes) == 1:
        axes = [axes]
    for idx, (a, b, dist) in enumerate(near_dupes[:3]):
        img_a = cv2.cvtColor(cv2.imread(a[3]), cv2.COLOR_BGR2RGB)
        img_b = cv2.cvtColor(cv2.imread(b[3]), cv2.COLOR_BGR2RGB)
        axes[idx][0].imshow(img_a)
        axes[idx][0].set_title(f"{a[1]}/{a[2]}", fontsize=8)
        axes[idx][0].axis('off')
        axes[idx][1].imshow(img_b)
        axes[idx][1].set_title(f"{b[1]}/{b[2]} (d={dist})", fontsize=8)
        axes[idx][1].axis('off')
    plt.suptitle("Near-duplicate examples", fontsize=12)
    plt.tight_layout()
    plt.show()
else:
    print("  Không tìm thấy near-duplicates trong sample")

## 4. Contrast & Brightness Analysis

In [ ]:
print("Đang tính brightness & contrast..")
brightness_data = []

for cls in tqdm(classes, desc="L-channel"):
    imgs = glob.glob(os.path.join(TRAIN_DIR, cls, "*.jpg"))[:SAMPLE_PER_CLASS]
    for p in imgs:
        img = cv2.imread(p)
        lab = cv2.cvtColor(img, cv2.COLOR_BGR2Lab)
        L = lab[:,:,0].astype(float)
        brightness_data.append({
            'class': cls,
            'brightness': L.mean(),
            'contrast': L.std()
        })

df_bc = pd.DataFrame(brightness_data)

print(f"\nBrightness: mean={df_bc['brightness'].mean():.1f}, std={df_bc['brightness'].std():.1f}")
print(f"Contrast:   mean={df_bc['contrast'].mean():.1f}, std={df_bc['contrast'].std():.1f}")

### Brightness & Contrast theo lớp

Sắp xếp 45 lớp theo brightness/contrast trung bình, chọn 5 thấp nhất + 5 cao nhất để so sánh.

In [ ]:
class_brightness = df_bc.groupby('class')['brightness'].mean().sort_values()
top_bottom = list(class_brightness.index[:5]) + list(class_brightness.index[-5:])

print("5 lớp tối nhất (brightness thấp):")
for cls in class_brightness.index[:5]:
    print(f"  {cls}: {class_brightness[cls]:.1f}")
print("5 lớp sáng nhất (brightness cao):")
for cls in class_brightness.index[-5:]:
    print(f"  {cls}: {class_brightness[cls]:.1f}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

df_sub = df_bc[df_bc['class'].isin(top_bottom)]
order = class_brightness[class_brightness.index.isin(top_bottom)].index.tolist()
sns.boxplot(data=df_sub, x='class', y='brightness', order=order, ax=axes[0], palette='coolwarm')
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=45, ha='right', fontsize=8)
axes[0].set_title("Brightness (5 tối nhất + 5 sáng nhất)")

class_contrast = df_bc.groupby('class')['contrast'].mean().sort_values()
top_bottom_c = list(class_contrast.index[:5]) + list(class_contrast.index[-5:])
df_sub_c = df_bc[df_bc['class'].isin(top_bottom_c)]
order_c = class_contrast[class_contrast.index.isin(top_bottom_c)].index.tolist()
sns.boxplot(data=df_sub_c, x='class', y='contrast', order=order_c, ax=axes[1], palette='coolwarm')
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=45, ha='right', fontsize=8)
axes[1].set_title("Contrast (5 thấp nhất + 5 cao nhất)")

plt.tight_layout()
plt.show()

In [ ]:
# Scatter plot brightness vs contrast - toàn bộ 45 lớp
fig, ax = plt.subplots(figsize=(10, 7))
class_bc_stats = df_bc.groupby('class').agg({'brightness': 'mean', 'contrast': 'mean'}).reset_index()
ax.scatter(class_bc_stats['brightness'], class_bc_stats['contrast'], s=60, alpha=0.7)
for _, row in class_bc_stats.iterrows():
    ax.annotate(row['class'], (row['brightness'], row['contrast']), fontsize=6, alpha=0.8)
ax.set_xlabel("Mean Brightness (L-channel)")
ax.set_ylabel("Mean Contrast (L-channel std)")
ax.set_title("Brightness vs Contrast theo lớp (toàn bộ 45 lớp)")
plt.tight_layout()
plt.show()

### Kiểm định: Brightness & Contrast khác biệt giữa lớp?

**H0:** Brightness (hoặc Contrast) trung bình không khác biệt giữa 45 lớp.

**H1:** Ít nhất 1 cặp lớp có Brightness (hoặc Contrast) khác biệt.

In [ ]:
# ANOVA + Eta-squared cho cả Brightness và Contrast
brightness_groups = [df_bc[df_bc['class']==c]['brightness'].values for c in classes]
contrast_groups = [df_bc[df_bc['class']==c]['contrast'].values for c in classes]

for metric_name, metric_groups in [("Brightness", brightness_groups), ("Contrast", contrast_groups)]:
    # Levene test
    lev_s, lev_p = stats.levene(*metric_groups)
    # ANOVA
    f_val, p_val = stats.f_oneway(*metric_groups)
    # Kruskal-Wallis
    h_val, kw_p = stats.kruskal(*metric_groups)
    # Eta-squared
    all_vals = np.concatenate(metric_groups)
    gm = all_vals.mean()
    ss_b = sum(len(g) * (np.mean(g) - gm)**2 for g in metric_groups)
    ss_t = np.sum((all_vals - gm)**2)
    eta2 = ss_b / ss_t if ss_t > 0 else 0

    print(f"--- {metric_name} ---")
    print(f"  Levene: stat={lev_s:.2f}, p={lev_p:.2e} -> Variance {'KHONG dong nhat' if lev_p < 0.05 else 'dong nhat'}")
    print(f"  ANOVA:  F={f_val:.2f}, p={p_val:.2e} -> {'BAC BO H0' if p_val < 0.05 else 'KHONG BAC BO H0'}")
    print(f"  Kruskal: H={h_val:.2f}, p={kw_p:.2e} -> {'BAC BO H0' if kw_p < 0.05 else 'KHONG BAC BO H0'}")
    print(f"  Eta² = {eta2:.3f} (effect size: {'LON' if eta2>0.14 else 'TRUNG BINH' if eta2>0.06 else 'NHO'})")
    print()

## 5. File Size & Kích thước ảnh

In [ ]:
print("Đang scan file size..")
file_sizes = []
img_dims = set()

for cls in tqdm(classes, desc="File size"):
    for p in glob.glob(os.path.join(TRAIN_DIR, cls, "*.jpg")):
        file_sizes.append(os.path.getsize(p))
    for p in glob.glob(os.path.join(TRAIN_DIR, cls, "*.jpg"))[:5]:
        img = cv2.imread(p)
        img_dims.add(img.shape[:2])

file_sizes_kb = np.array(file_sizes) / 1024

print(f"\nFile size: mean={file_sizes_kb.mean():.1f}KB, std={file_sizes_kb.std():.1f}KB")
print(f"  min={file_sizes_kb.min():.1f}KB, max={file_sizes_kb.max():.1f}KB")
print(f"Kích thước ảnh unique: {img_dims}")
if len(img_dims) == 1:
    print("  Tất cả ảnh cùng kích thước")
else:
    print("  Phát hiện ảnh có kích thước khác nhau!")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(file_sizes_kb, bins=80, color='#4C72B0', edgecolor='white', alpha=0.8)
ax.axvline(file_sizes_kb.mean(), color='red', linestyle='--', label=f'Mean={file_sizes_kb.mean():.1f}KB')
ax.set_xlabel("File size (KB)")
ax.set_ylabel("Số ảnh")
ax.set_title("Phân bố file size")
ax.legend()
plt.tight_layout()
plt.show()

q1, q3 = np.percentile(file_sizes_kb, [25, 75])
iqr = q3 - q1
lower, upper = q1 - 1.5*iqr, q3 + 1.5*iqr
outliers = file_sizes_kb[(file_sizes_kb < lower) | (file_sizes_kb > upper)]
print(f"File size outliers (IQR): {len(outliers)}/{len(file_sizes_kb)} ảnh ({len(outliers)/len(file_sizes_kb)*100:.1f}%)")

## 6. Tổng kết EDA

In [ ]:
print("=" * 60)
print("TONG KET EDA - NWPU-RESISC45")
print("=" * 60)
print(f"- Tong anh: {df_counts['total'].sum():,} ({df_counts['train'].sum():,} train / {df_counts['test'].sum():,} test)")
print(f"- So lop: {len(classes)}")
print(f"- Kich thuoc: {img_dims}")
print(f"- Class balance: {'Can bang' if imbalance_ratio <= 1.1 else f'Chenh lech {imbalance_ratio:.1f}x'}")
print(f"- Pixel mean (R,G,B): ({np.mean(all_means_r):.1f}, {np.mean(all_means_g):.1f}, {np.mean(all_means_b):.1f})")
print(f"- Pixel ANOVA: F={f_stat:.2f}, Eta²={eta_sq_pixel:.3f} -> Khac biet CO Y NGHIA giua lop")
print(f"- pHash exact dupes: {len(exact_dupes)} nhom")
print(f"- Near-dupes (sample): {len(near_dupes)} cap")
print(f"- Brightness/Contrast: Khac biet CO Y NGHIA giua lop (ANOVA p<0.05)")
print(f"- File size: {file_sizes_kb.mean():.1f} +/- {file_sizes_kb.std():.1f} KB")
print("=" * 60)